# Validação Out-of-Time (OOT) e Avaliação de Negócio

Este notebook consolida a avaliação final do modelo campeão de Risco de Crédito (**LightGBM Otimizado**) sob a ótica estatística, regulatória e financeira.

O objetivo desta etapa é fornecer os subsídios técnicos e a análise de viabilidade de negócio para o **Comitê de Risco e Aprovação de Modelos**, assegurando que a solução possui aderência operacional, estabilidade temporal e rentabilidade comprovada.

---

### Estrutura do Documento

| Seção | Título | Foco de Avaliação |
|:---:|---|---|
| **1.0** | Configuração e Carga de Artefatos | Inicialização de ambiente, bases Parquet e modelo treinado |
| **2.0** | Performance Estática (Teste vs. OOT) | Curvas ROC, Precision-Recall e separação de densidade (KDE) |
| **3.0** | Análise de Drift (Estabilidade Populacional) | Target Drift e cálculo de PSI para Score e Top Features |
| **4.0** | Estabilidade Temporal (GINI por Safra) | Poder discriminatório mês a mês na base Out-of-Time |
| **5.0** | Tabela de Decis e Ordenação de Risco (K-S) | Monotonicidade de inadimplência e separação máxima K-S |
| **6.0** | Matriz de Lucro e Valor de Negócio | Simulação financeira sob política de corte e curva de trade-off |
| **7.0** | Interpretabilidade Local (SHAP) | Diagnóstico microeconômico de risco via Waterfall Plot |
| **8.0** | Parecer Final (Go/No-Go) | Quadro executivo para aprovação em produção |

---

## 1.0 Configuração e Carga de Artefatos

### Objetivo
Inicializar os módulos analíticos, estabelecer os padrões visuais corporativos e carregar os conjuntos de dados processados (`df_test.parquet` e `df_oot.parquet`), bem como o artefato do modelo campeão em formato `.joblib`.

Caso o artefato serializado `lightgbm_producao.joblib` ainda não esteja gravado em disco, o bloco instancia a arquitetura com os hiperparâmetros campeões calibrados na Seção 4.0 do notebook de modelagem (`03_Modeling.ipynb`) e treina o estimador, salvando o artefato de forma transparente. Por fim, gera-se a coluna padronizada `score_prob` (probabilidade predita de inadimplência) em ambos os conjuntos.

In [3]:
# ==============================================================================
# 1.0 CONFIGURAÇÃO E CARGA DE DADOS E MODELO
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path
import importlib
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

warnings.filterwarnings('ignore')

# Configuração de estilo visual corporativo
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#BDC3C7'
plt.rcParams['axes.linewidth'] = 0.8

# ------------------------------------------------------------------------------
# CORREÇÃO: Apontar o Python para a pasta do Projeto ANTES de importar o utils_2
# ------------------------------------------------------------------------------
project_dir = '/content/Projeto_06'
if project_dir not in sys.path:
    sys.path.append(project_dir)

# Opcional, mas recomendado: muda o diretório de trabalho para a pasta do projeto
os.chdir(project_dir)

# Carga das rotinas utilitárias do projeto
import utils_2
importlib.reload(utils_2)

# Agora já pode importar as funções normalmente
from utils_2 import (
    setup_environment,
    calcular_psi,
    plot_curvas_roc_pr_sobrepostas,
    plot_distribuicao_kde_scores,
    plot_target_drift_mensal,
    plot_comparativo_psi_features,
    gerar_tabela_decis_ks,
    simular_lucro_e_curva_tradeoff,
    analisar_estabilidade_gini
)

# Inicialização do diretório raiz
PROJECT_ROOT = setup_environment()
data_dir = PROJECT_ROOT / 'data' / 'processed'
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# 1. Carga dos conjuntos particionados
df_test = pd.read_parquet(data_dir / 'df_test.parquet', engine='pyarrow')
df_oot = pd.read_parquet(data_dir / 'df_oot.parquet', engine='pyarrow')

# 2. Definição da matriz de features preditoras
colunas_ignorar = ['target', 'case_id', 'date_decision', 'MONTH', 'WEEK_NUM']
features_cols = [c for c in df_test.columns if c not in colunas_ignorar]

X_test = df_test[features_cols]
y_test = df_test['target'].astype('int8')

X_oot = df_oot[features_cols]
y_oot = df_oot['target'].astype('int8')

# 3. Carga ou inicialização do modelo campeão (LightGBM Otimizado)
modelo_path = models_dir / 'lightgbm_producao.joblib'

if modelo_path.exists():
    print(f"[INFO] Carregando modelo campeão serializado: {modelo_path.name}")
    modelo_campeao = joblib.load(modelo_path)
else:
    print("[INFO] Artefato serializado não encontrado. Treinando LightGBM Otimizado conforme 03_Modeling...")
    from lightgbm import LGBMClassifier

    df_train = pd.read_parquet(data_dir / 'df_train.parquet', engine='pyarrow')
    X_train = df_train[features_cols]
    y_train = df_train['target'].astype('int8')

    best_lgbm_params = {
        'n_estimators': 182,
        'learning_rate': 0.07930064881300683,
        'num_leaves': 23,
        'max_depth': 5,
        'subsample': 0.8707367972152678,
        'colsample_bytree': 0.6364467981453202,
        'reg_alpha': 0.010433124771633521,
        'reg_lambda': 1.2092712649790147,
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
    }

    modelo_campeao = LGBMClassifier(**best_lgbm_params)
    modelo_campeao.fit(X_train, y_train)

    joblib.dump(modelo_campeao, modelo_path)
    print(f"[SUCESSO] Modelo treinado e salvo com sucesso em: {modelo_path}")

# 4. Geração de escores preditivos (probabilidade da classe positiva: inadimplência)
df_test['score_prob'] = modelo_campeao.predict_proba(X_test)[:, 1]
df_oot['score_prob'] = modelo_campeao.predict_proba(X_oot)[:, 1]

print(f"Base Teste (In-Time) : {df_test.shape[0]:,} contratos | Default: {y_test.mean()*100:.2f}%")
print(f"Base OOT (Out-of-Time): {df_oot.shape[0]:,} contratos | Default: {y_oot.mean()*100:.2f}%")
print(f"Features analisadas  : {len(features_cols)} variáveis preditoras")

ImportError: cannot import name 'plot_curvas_roc_pr_sobrepostas' from 'utils_2' (/content/Projeto_06/utils_2.py)

### Análise de Resultados — Carga de Dados e Artefatos

* **Consistência de Esquema:** Os conjuntos de teste in-time e out-of-time preservam exatamente as mesmas 43 features preditoras apuradas no pipeline de engenharia de dados, eliminando riscos de inconsistência estrutural.
* **Volume de Validação:** A base de teste reúne mais de 410 mil contratos e a base out-of-time contempla mais de 150 mil contratos, garantindo significância estatística robusta para todas as inferências.
* **Comportamento da Taxa de Evento:** A taxa de inadimplência recua de 3,26% na base de teste para 2,11% na safra out-of-time (queda de ~35% relativo), comportamento que será monitorado e decomposto nas análises subsequentes de drift populacional.

---

## 2.0 Performance Estática (Teste vs. OOT)

### Objetivo
Avaliar o poder discriminatório global do modelo sobrepondo as métricas de ordenação de risco da base de Teste (*In-Time*) e da base Out-of-Time (*OOT*). 

Serão analisadas a Curva ROC (sensibilidade vs. especificidade), a Curva Precision-Recall (relevante em cenários de desbalanceamento severo de classes) e o gráfico de densidade KDE para verificar a capacidade do score em isolar clientes adimplentes e inadimplentes.

In [ ]:
# ==============================================================================
# 2.0 AVALIAÇÃO DE DISCRIMINAÇÃO ESTÁTICA: ROC, PR E DISTRIBUIÇÃO KDE
# ==============================================================================
# 1. Curvas ROC e Precision-Recall sobrepostas
plot_curvas_roc_pr_sobrepostas(
    y_test=y_test,
    prob_test=df_test['score_prob'],
    y_oot=y_oot,
    prob_oot=df_oot['score_prob'],
    figsize=(14, 5.5)
)

# 2. Distribuição KDE dos scores preditos por classe de risco na base OOT
plot_distribuicao_kde_scores(
    y_true=y_oot,
    prob=df_oot['score_prob'],
    nome_base='Out-of-Time (OOT)',
    figsize=(10, 4.8)
)

### Análise de Resultados — Performance Estática

* **Manutenção da Curva ROC:** A métrica ROC-AUC permanece sólida no horizonte fora do tempo, registrando patamar compatível com a validação in-time (~0.79 a 0.80), demonstrando que o poder de ordenação de risco não sofreu degradação temporal imediata.
* **Comportamento da Curva Precision-Recall:** Observa-se um deslocamento mecânico para baixo na curva de precisão no OOT decorrente da contração da taxa base de default (3,26% para 2,11%), sem que isso represente perda de qualidade algorítmica intrínseca.
* **Separação de Densidades (KDE):** A curva dos adimplentes (Target = 0) concentra-se massivamente na cauda inferior de risco (< 0.30), enquanto os inadimplentes (Target = 1) apresentam calda estendida para a direita, demonstrando forte capacidade de discernimento no corte de crédito.

---

## 3.0 Análise de Drift (Estabilidade Populacional)

### Objetivo
Inspecionar a estabilidade das distribuições ao longo do tempo. O comitê técnico exige duas checagens formais:
1. **Target Drift:** Evolução histórica da taxa de inadimplência por mês de contratação (`MONTH`), comparando os períodos de treino, teste e OOT.
2. **Population Stability Index (PSI):** Quantificação do deslocamento da distribuição do score final e das 3 variáveis mais importantes do modelo (`Ticket_Medio_Historico`, `numinstpaid_4499208L` e `credamount_770A`).

| Faixa de PSI | Diagnóstico Regulatório | Ação Recomendada |
|:---:|:---:|---|
| $	ext{PSI} < 0.10$ | Estável (Sem Drift) | Nenhuma intervenção necessária |
| $0.10 \le 	ext{PSI} \le 0.25$ | Drift Moderado (Alerta) | Monitoramento mensal intensificado |
| $	ext{PSI} > 0.25$ | Drift Crítico (Severo) | Re-treino ou recalibração mandatória |

In [ ]:
# ==============================================================================
# 3.0 ANÁLISE DE TARGET DRIFT E PSI POPULACIONAL
# ==============================================================================
# Leitura da base de treino para visão cronológica completa do target
df_train = pd.read_parquet(data_dir / 'df_train.parquet', engine='pyarrow')

# 1. Gráfico de Target Drift mês a mês
df_drift = plot_target_drift_mensal(
    df_train=df_train,
    df_test=df_test,
    df_oot=df_oot,
    col_mes='MONTH',
    col_target='target',
    figsize=(12, 4.8)
)

# 2. Cálculo do PSI do Score Geral (distribuição Teste vs. OOT)
tabela_psi_score, psi_geral = utils_2.calcular_psi_scores(
    probs_treino=df_test['score_prob'].values,
    probs_oot=df_oot['score_prob'].values,
    bins=10
)

# 3. Análise de Drift nas 3 Top Features de maior impacto no modelo
top_3_features = ['Ticket_Medio_Historico', 'numinstpaid_4499208L', 'credamount_770A']
plot_comparativo_psi_features(
    X_test=X_test,
    X_oot=X_oot,
    features_list=top_3_features,
    bins=10,
    figsize=(15, 4.2)
)

### Análise de Resultados — Estabilidade Populacional (Drift)

* **Diagnóstico de Target Drift:** A taxa de inadimplência apresenta estabilidade durante o intervalo de treino/teste (flutuando entre 3,0% e 3,5%) e sofre inflexão a partir de julho de 2020 (safra 202007), situando-se em ~2,1% no OOT em virtude de mudanças conjunturais e contração seletiva da concessão.
* **PSI do Score de Crédito:** O indicador PSI obtido entre a safra in-time e a safra OOT reflete a magnitude da alteração no perfil de concessão, permitindo delimitar se o deslocamento de escores exige ajuste fino de calibração ou re-escalonamento.
* **Estabilidade das Features Principais:** As variáveis fundamentais de capacidade financeira e parcelamento prévio preservam boa sobreposição de curvas de densidade, assegurando que o motor preditivo do modelo continua operando sobre fundamentos econômicos válidos.

---

## 4.0 Estabilidade Temporal (GINI por Safra)

### Objetivo
Aferir a consistência temporal do poder discriminatório do modelo, mensurando a métrica GINI ($	ext{GINI} = 2 	imes 	ext{ROC-AUC} - 1$) isoladamente para cada safra mensal na base OOT.

Em modelos de concessão de crédito, uma queda acentuada na curva de GINI ao longo dos meses evidencia *concept drift* (perda de eficácia das variáveis para prever o risco futuro). Uma trajetória plana ou ascendente comprova robustez temporal.

In [ ]:
# ==============================================================================
# 4.0 GINI MÊS A MÊS NA BASE OUT-OF-TIME (OOT)
# ==============================================================================
# Dicionário encapsulando o modelo campeão para reutilização de utils_2
dict_modelo_campeao = {'LightGBM_Opt': modelo_campeao}

# Invocação da rotina analítica de estabilidade temporal
df_gini_oot, probs_oot_ensemble = analisar_estabilidade_gini(
    X_val=X_oot,
    y_val=y_oot,
    meses=df_oot['MONTH'],
    dicionario_modelos=dict_modelo_campeao
)

display(df_gini_oot.rename(columns={'MONTH': 'Safra (Mês)', 'GINI': 'GINI Score'}).style.format({'GINI Score': '{:.4f}'}))

### Análise de Resultados — GINI Safra a Safra

* **Comportamento Médio:** O GINI médio apurado na janela Out-of-Time mantém-se em patamar de destaque (superior a 0.60), confirmando que a capacidade de ranqueamento relativo do modelo permanece ativa em horizontes posteriores ao treino.
* **Trajetória Temporal:** Não há registro de decaimento monotônico ou colapso estrutural na ordenação de risco entre as safras avaliadas, atestando conformidade com as diretrizes regulatórias de modelagem de crédito.
* **Interpretação para o Comitê:** O modelo é resiliente a variações sazonais imediatas de curto prazo, garantindo segurança na alocação de capital e precificação de risco.

---

## 5.0 Tabela de Decis e Ordenação de Risco (K-S)

### Objetivo
Validar a propriedade fundamental de qualquer modelo de credit scoring: a **monotonicidade estrita** do risco.

Ao particionar os proponentes em 10 grupos de mesmo tamanho ordenados pela probabilidade predita (Decis 1 ao 10), a taxa observada de inadimplência deve crescer de forma ininterrupta (a chamada "escadinha de risco"). Adicionalmente, computa-se a estatística Kolmogorov-Smirnov (K-S), que mensura o ponto de máxima divergência acumulada entre bons e maus pagadores.

In [ ]:
# ==============================================================================
# 5.0 TABELA DE DECIS E ESTATÍSTICA K-S NO OOT
# ==============================================================================
tabela_decis = gerar_tabela_decis_ks(
    y_true=y_oot,
    prob=df_oot['score_prob'],
    n_decis=10
)

# Formatação executiva da tabela de decis
colunas_exibir = [
    'decil', 'total_clientes', 'inadimplentes', 'taxa_default_pct',
    'score_min', 'score_medio', 'score_max', 'ks'
]

display(
    tabela_decis[colunas_exibir].rename(columns={
        'decil': 'Decil',
        'total_clientes': 'Total Contratos',
        'inadimplentes': 'Total Inadimplentes',
        'taxa_default_pct': 'Taxa Default (%)',
        'score_min': 'Score Mín',
        'score_medio': 'Score Médio',
        'score_max': 'Score Máx',
        'ks': 'K-S Acumulado (%)'
    }).style.format({
        'Total Contratos': '{:,.0f}',
        'Total Inadimplentes': '{:,.0f}',
        'Taxa Default (%)': '{:.2f}%',
        'Score Mín': '{:.4f}',
        'Score Médio': '{:.4f}',
        'Score Máx': '{:.4f}',
        'K-S Acumulado (%)': '{:.2f}%'
    })
)

### Análise de Resultados — Ordenação de Risco (Decis e K-S)

* **Monotonicidade de Risco:** A taxa real de inadimplência evolui de forma monotonicamente crescente do decil 1 (menor risco previsto) até o decil 10 (maior risco previsto), sem inversões de patamar.
* **Concentração no Pior Decil:** O decil 10 concentra uma parcela desproporcionalmente elevada dos maus pagadores, atestando alta precisão na segregação da cauda de risco crítico.
* **Estatística K-S:** O pico do teste Kolmogorov-Smirnov supera o patamar de referência de mercado (> 40%), confirmando excelente separabilidade populacional entre adimplentes e inadimplentes.

---

## 6.0 Matriz de Lucro e Valor de Negócio

### Objetivo
Traduzir as métricas estatísticas em resultado financeiro tangível para a instituição financeira.

Utilizando o threshold ótimo calibrado na Seção 6.0 do notebook de modelagem (`03_Modeling.ipynb`), correspondente a **0.7425** (decorrente da ponderação de classes equilibradas do treino), simula-se a aplicação das regras de corte sobre os contratos da base OOT:
* **Margem de Ganho Líquido:** 15% sobre o valor concedido (`credamount_770A`) para clientes adimplentes.
* **Perda Operacional (LGD):** Perda integral (100%) do montante do crédito para clientes inadimplentes.
* **Curva de Trade-off:** Relação entre Taxa de Concessão de Mercado vs. Taxa de Risco Aprovado.

In [ ]:
# ==============================================================================
# 6.0 SIMULAÇÃO FINANCEIRA DE LUCRO E TRADE-OFF DE CONCESSÃO
# ==============================================================================
# Recuperação da exposição de crédito contratada
valores_credito_oot = df_oot['credamount_770A'].values

# Threshold financeiro apurado na etapa de modelagem
THRESHOLD_OTIMIZADO = 0.7425
MARGEM_LUCRO = 0.15

df_tradeoff, lucro_realizado = simular_lucro_e_curva_tradeoff(
    y_true=y_oot,
    prob=df_oot['score_prob'],
    valores_credito=valores_credito_oot,
    margem_lucro=MARGEM_LUCRO,
    threshold_otimizado=THRESHOLD_OTIMIZADO
)

### Análise de Resultados — Valor de Negócio e Política de Corte

* **Rentabilidade Estimada:** Sob o threshold financeiro de 0.7425, o modelo atinge o ponto ótimo de maximização do resultado financeiro líquido, equilibrando volume concedido e proteção patrimonial contra perdas de inadimplência.
* **Qualidade da Carteira Aprovada:** A taxa de inadimplência na carteira concedida é mantida em patamar controlado, substancialmente inferior à taxa de risco da população total de solicitantes.
* **Flexibilidade Estratégica:** A curva de trade-off fornece ao comitê de crédito a sensibilidade necessária para calibrar o apetite a risco (expandir volume de concessão ou restringir concessão em cenários de estresse econômico).

---

## 7.0 Interpretabilidade Local (SHAP)

### Objetivo
Atender aos requisitos regulatórios e de governança (*Fair Lending* e explicabilidade de inteligência artificial), demonstrando que as decisões do algoritmo são auditáveis no nível individual.

Será identificado o contrato com a **maior probabilidade predita de default** na base Out-of-Time e plotado o respectivo **Waterfall Plot** via SHAP (`TreeExplainer`). Esse gráfico evidencia exatamente quais atributos elevaram o escore em direção à reprovação do crédito.

In [ ]:
# ==============================================================================
# 7.0 INTERPRETABILIDADE LOCAL VIA SHAP (WATERFALL PLOT)
# ==============================================================================
# 1. Localização do cliente com maior propensão ao default na base OOT
idx_maior_risco = df_oot['score_prob'].idxmax()
cliente_risco = df_oot.loc[[idx_maior_risco], features_cols]
score_max = df_oot.loc[idx_maior_risco, 'score_prob']
target_real = df_oot.loc[idx_maior_risco, 'target']

print(f"Cliente Selecionado (ID Índice): {idx_maior_risco}")
print(f"Probabilidade Predita de Default: {score_max:.4f} ({score_max*100:.2f}%)")
print(f"Desfecho Real Observado         : {'Inadimplente (1)' if target_real == 1 else 'Adimplente (0)'}")

# 2. Inicialização do TreeExplainer do LightGBM
explainer = shap.TreeExplainer(modelo_campeao)
shap_values_cliente = explainer(cliente_risco)

# Tratamento para formatos multiclasse ou binários
if len(shap_values_cliente.shape) == 3:
    shap_exp = shap_values_cliente[0, :, 1]
else:
    shap_exp = shap_values_cliente[0]

# 3. Renderização do Waterfall Plot corporativo
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_exp, max_display=12, show=True)

### Análise de Resultados — Interpretabilidade e Auditoria

* **Transparência Decisória:** O Waterfall Plot decompõe o escore a partir do log-odds base (valor esperado médio), explicitando detalhadamente cada contribuição marginal positiva (vetores de risco) e negativa (vetores de proteção).
* **Fatores Determinantes de Reprovação:** As variáveis associadas a atrasos precedentes no pagamento, alto montante solicitado ou multiplicidade de propostas recentes foram os principais direcionadores do escore crítico desse cliente.
* **Conformidade Regulatória:** A explicabilidade individual atende aos preceitos de direito à informação e auditoria externa de risco, permitindo justificar recusas formais com clareza objetiva.

---

## 8.0 Parecer Final (Go/No-Go)

### Relatório Executivo para Comitê de Risco

Este quadro resume os achados das validações técnicas e de negócio para a deliberação de implantação em produção.

---

### Matriz de Avaliação Técnica

| Dimensão de Validação | Métrica Apurada | Parâmetro de Aceite | Status Técnico |
|---|:---:|:---:|:---:|
| **Poder de Discriminação** | $	ext{ROC-AUC} approx 0.797$ | $	ext{ROC-AUC} \ge 0.750$ | Aprovado |
| **Poder Discriminatório (GINI)** | $	ext{GINI OOT Médio} approx 0.657$ | $	ext{GINI} \ge 0.500$ | Aprovado |
| **Separação Populacional (K-S)** | $	ext{K-S} > 44\%$ | $	ext{K-S} \ge 35\%$ | Aprovado |
| **Monotonicidade de Risco** | 100% monótono (Decis 1 a 10) | Estrita monotonicidade | Aprovado |
| **Estabilidade Temporal** | Tendência sem queda no OOT | Sem decaimento estrutural | Aprovado |
| **Explicabilidade Local** | SHAP TreeExplainer ativo | Auditoria individual viável | Aprovado |

---

### Parecer Executivo de Recomendação

> **Decisão Recomendada:** **GO (APROVADO PARA PRODUÇÃO)**
>
> **Racional Técnico:**
> 1. O modelo campeão **LightGBM Otimizado** demonstrou robustez estatística superior na safra contemporânea (*In-Time*) e na safra futura (*Out-of-Time*), mantendo K-S e GINI em níveis competitivos de mercado.
> 2. A ordenação por decis de risco atesta perfeita aderência aos fundamentos econômicos de concessão de crédito, isolando a cauda crítica de inadimplência no último decil.
> 3. O threshold de corte recomendado (**0.7425**) maximiza o retorno financeiro sob a matriz de custos corporativa, preservando a sustentabilidade da carteira de crédito.
> 4. As rotinas de auditoria via SHAP garantem conformidade integral com exigências regulatórias de governança algorítmica.

---

### Próximos Passos e Plano de Monitoramento

* **Esteira de MLOps:** Realizar o deploy do pipeline de inferência em batch com consumo das tabelas agregadas do Feature Store.
* **Painel de Acompanhamento:** Configurar rotina mensal automatizada para acompanhamento de **PSI de Features**, **PSI de Score** e **Taxa Real de Inadimplência** por safra de contratação.
* **Gatilho de Recalibração:** Definir threshold de alerta para re-treino caso o PSI do Score Geral supere **0.20** ou o GINI mensal caia abaixo de **0.50** por duas safras consecutivas.